# The chip panel: what is in it, and what it shows

This notebook reads a panel made by `assemble_period.py` (README A7) and shows, in order:

1. what the panel is,
2. **the panel as a table**: real rows with every variable, chips followed through time,
   and what each variable means, with its distribution,
3. whether it passes its checks,
4. how many chips AMW's rules detect, per month and per quarter,
5. how well each period was seen through the clouds,
6. whether detections depend on how well a chip was seen,
7. how the months agree with their quarter,
8. where the detections are,
9. how much AMW's distances differ from distances on the Earth,
10. one chip followed through time.

Run it from the top after assembling both frequencies. It reads a few columns of one
period at a time, so a laptop can run it on the panel of subregions 2 and 3 (46 million
chips per period).

In [ ]:
# ---- settings: the only cell you normally edit ---------------------------------------
# On Goethe, cluster/notebook.sbatch sets these from the command line, e.g. MAP_PERIOD=2026Q3;
# without them (a laptop) the values after `or` are used.
import os
SUBREGIONS_SET = os.environ.get("SUBREGIONS", "2 3")   # "all" = the whole region's panel

RUN = os.environ.get("RUN") or "tests_output/aca_full_2026-10-23"   # the run folder, from the repository root
SUBREGIONS = None if SUBREGIONS_SET == "all" else [int(n) for n in SUBREGIONS_SET.split()]   # [2, 3]: the panel made with --subregions 2 3; None = the whole region
RUN_CHECKS = os.environ.get("RUN_CHECKS", "1") != "0"   # section 3 reads every period once (about half a minute each)
MAP_PERIOD = os.environ.get("MAP_PERIOD") or None   # the quarter shown in sections 2 and 8 to 10, e.g. "2025Q2"; None = the last assembled one
EXPORT_DIR = os.environ.get("EXPORT_DIR") or "tests_output/panel_tables"   # section 2 also saves its tables here as CSV (opens in Excel); None = don't

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.compute as pc
import pyarrow.dataset as ds
import pyarrow.parquet as pq

# The repository root: the first folder upwards that holds panel_pipeline/.
REPO = Path.cwd().resolve()
while not (REPO / "panel_pipeline").is_dir():
    if REPO.parent == REPO:
        raise SystemExit("Start Jupyter inside the AMW-mining-detector repository")
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "panel_pipeline" / "lib"))      # periods.py, read_panel.py
sys.path.insert(0, str(REPO / "panel_pipeline" / "checks"))   # check_panel.py
from periods import Period
from read_panel import panel_dir
from check_panel import check_panel

run = (REPO / RUN).resolve()
panel = panel_dir(run, SUBREGIONS)
pd.set_option("display.width", 160, "display.max_columns", 30)

MONTHLY, QUARTERLY = "#2a78d6", "#eb6834"      # one colour per panel, the same in every chart
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#8a8984", "axes.labelcolor": "#52514e",
                     "xtick.color": "#52514e", "ytick.color": "#52514e",
                     "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6,
                     "axes.axisbelow": True, "figure.dpi": 110})


def tags(frequency: str) -> list[str]:
    # The periods of one frequency whose two layers are both written.
    return sorted(p.name.removeprefix("period=")
                  for p in (panel / frequency / "postprocess").glob("period=*")
                  if (p / "_record.json").exists())


def record(frequency: str, layer: str, tag: str) -> dict:
    return json.loads((panel / frequency / layer / f"period={tag}" / "_record.json").read_text())


def column(frequency: str, layer: str, tag: str, name: str) -> np.ndarray:
    # One column of one period as a numpy array. A flag's "no score" (missing) reads as
    # False here; use the `scored` column to tell the two apart.
    a = pq.read_table(panel / frequency / layer / f"period={tag}" / "data.parquet",
                      columns=[name]).column(name)
    if a.type == "bool":
        a = a.fill_null(False)
    return a.to_numpy()


periods = {f: tags(f) for f in ("monthly", "quarterly")}
print(panel)
for f, t in periods.items():
    print(f"  {f:9s} {len(t)} period(s): {', '.join(t) if t else 'none yet'}")
if MAP_PERIOD is None:                        # the last quarter that is assembled
    MAP_PERIOD = periods["quarterly"][-1] if periods["quarterly"] else None
print(f"quarter shown in sections 2 and 8 to 10: {MAP_PERIOD}")

## 1. What the panel is

The panel has **one row per chip and period**. A chip is AMW's 48 × 48 pixel square
(480 m), the model's input; chips overlap by half, 529 per tile.

* Two panels, **monthly** and **quarterly**, in two folders. They share `chip_id`, so a month
  can be joined to its quarter.
* Each period has two **layers**, two files with exactly the same rows in the same order
  (every planned chip once, sorted by `chip_id`):
  `scores` holds what `detect.py` measured (the model's six votes, their mean `confidence`,
  and how well the period was seen); `postprocess` holds AMW's detection flags and the
  distances they rest on. Later steps add layers; nothing written is ever changed.
* What does not change over time (the chip's square, its tile, whether it lies in the
  Andes) is stored once, in `plan/chips.parquet`.
* A chip whose tile could not be scored keeps its row, with `scored = False` and empty
  values, so "no data" never reads as "no mine".

In [ ]:
rows = []
for f, ts in periods.items():
    for t in ts:
        s, p = record(f, "scores", t), record(f, "postprocess", t)
        rows.append({"frequency": f, "period": t, "chips": s["rows"], "scored": s["scored"],
                     "subregions": s["subregions"],
                     "border chips": s["border"]["chips_scored_twice"],
                     "copies differ": s["border"]["copies_differ"],
                     "size MB": round(sum((panel / f / layer / f"period={t}" / "data.parquet").stat().st_size
                                          for layer in ("scores", "postprocess")) / 1e6)})
overview = pd.DataFrame(rows)
print("scores columns:     ", ", ".join(s["columns"]))
print("postprocess columns:", ", ".join(p["columns"]))
overview

**Border chips.** A tile on the border between two subregions is scored once by each, under the
same `chip_id`. The panel keeps one row per chip, with the higher score as AMW does; the
table counts how often the two scores differed (in this run: never).

## 2. The panel as a table

This is the data set the econometric model reads. Its **unit of observation is a chip in a
period**: `chip_id` identifies the chip (the cross-section), `period` the month or quarter
(the time dimension). The panel is **balanced**: every planned chip has a row in every
period, scored or not, so nothing drops out of the sample because it was cloudy.

A row is put together from three files that line up on `chip_id`:

| part | file | holds |
|---|---|---|
| the chip | `plan/chips.parquet` | where it is and what never changes (square, tile, Andes, in the region) |
| `scores` layer | `<panel>/<frequency>/scores/period=<tag>/data.parquet` | the model's score and how well the period was seen |
| `postprocess` layer | `<panel>/<frequency>/postprocess/period=<tag>/data.parquet` | AMW's detection rules and the distances they use |

The cells below show, in order: real rows of one quarter with every variable; a few chips
followed through every period in the long format a regression reads; what each variable
means; and how each variable is distributed in one quarter and one month. If `EXPORT_DIR`
is set, the four tables are also saved as CSV files to send around.

In [ ]:
import pyarrow as pa
from variables import dictionary, expand

PLAN_COLUMNS = ["chip_id", "tile_key", "chip_index", "minx", "miny", "maxx", "maxy", "in_region", "in_andes"]


def layer_path(frequency: str, layer: str, tag: str) -> Path:
    return panel / frequency / layer / f"period={tag}" / "data.parquet"


def rows_at(path: Path, positions: np.ndarray, columns: list[str] | None = None) -> pd.DataFrame:
    # Rows at these positions of one layer file, reading only the row groups that hold them,
    # so a handful of rows of a 46-million-row period costs a fraction of a second.
    f = pq.ParquetFile(path)
    starts = np.concatenate([[0], np.cumsum([f.metadata.row_group(i).num_rows
                                             for i in range(f.num_row_groups)])])
    positions = np.unique(positions)
    groups = np.searchsorted(starts, positions, side="right") - 1
    parts = [f.read_row_group(int(g), columns=columns).to_pandas()
              .iloc[positions[groups == g] - starts[g]].set_axis(positions[groups == g])
             for g in np.unique(groups)]
    return pd.concat(parts)


def panel_rows(frequency: str, tag: str, positions: np.ndarray,
               columns: dict[str, list[str]] | None = None) -> pd.DataFrame:
    # Both layers of one period at these row positions, side by side, with `period` added.
    layers = []
    for layer in ("scores", "postprocess"):
        cols = None if columns is None else ["chip_id", *columns.get(layer, [])]
        layers.append(rows_at(layer_path(frequency, layer, tag), positions, cols))
    assert layers[0]["chip_id"].equals(layers[1]["chip_id"]), "the layers' rows don't line up"
    out = pd.concat([layers[0], layers[1].drop(columns="chip_id")], axis=1)
    out.insert(1, "period", tag)
    return out


def chip_facts(ids) -> pd.DataFrame:
    # What never changes about these chips, from the plan, with the square's centre.
    t = ds.dataset(run / "plan" / "chips.parquet").to_table(
        columns=PLAN_COLUMNS, filter=pc.field("chip_id").isin(pa.array(np.asarray(ids)))).to_pandas()
    t["lon"], t["lat"] = (t["minx"] + t["maxx"]) / 2, (t["miny"] + t["maxy"]) / 2
    return t


# Which rows to show: in MAP_PERIOD, the strongest stringent detections, random scored chips,
# and (if there are any) a chip with no clear picture and a chip whose tile failed.
q = MAP_PERIOD
scored_q = column("quarterly", "scores", q, "scored")
conf_q = column("quarterly", "scores", q, "confidence")
strict_q = column("quarterly", "postprocess", q, "det_stringent")
n_clear_q = column("quarterly", "scores", q, "n_clear")
rng = np.random.default_rng(0)
top = [p for p in np.argsort(np.where(strict_q, conf_q, -1))[::-1][:3] if strict_q[p]]
picks = {**{p: "strongest stringent detection" for p in top},
         **{p: "random scored chip" for p in rng.choice(np.flatnonzero(scored_q), 3, replace=False)}}
for p, why in ((np.flatnonzero(n_clear_q == 0)[:1], "no clear picture (n_clear = 0)"),
               (np.flatnonzero(~scored_q)[:1], "tile failed (not scored)")):
    picks.update({int(x): why for x in p})
positions = np.array(sorted(picks, key=list(picks).index))

rows = panel_rows("quarterly", q, positions)
facts = chip_facts(rows["chip_id"])
example = rows.merge(facts, on="chip_id", how="left")
example.insert(0, "shown because", [picks[p] for p in rows.index])
example = example[["shown because", "chip_id", "period", "lon", "lat", "in_region", "in_andes",
                   *[c for c in rows.columns if c not in ("chip_id", "period")],
                   "tile_key", "chip_index", "minx", "miny", "maxx", "maxy"]]
print(f"{len(example)} of the {len(scored_q):,} rows of the quarterly panel, {q} "
      f"(scroll right: {example.shape[1] - 1} variables)")
with pd.option_context("display.max_columns", None, "display.width", 250):
    display(example)

**The same chips through time.** The model reads the panel in *long* format: one row per
chip and period, stacked. Because every period's file holds the same chips in the same order,
a chip's row is at the same position in every file, so this needs no join. A selection of
variables, for the first three chips above, in every assembled month and quarter:

In [ ]:
LONG = {"scores": ["scored", "confidence", "n_obs", "n_clear", "empty_share", "cs_mean", "ndvi_mean", "bsi_mean"],
        "postprocess": ["det_raw", "det_loose", "det_stringent"]}
follow = positions[:3]
history = []
for f, ts in periods.items():
    for t in ts:
        part = panel_rows(f, t, follow, {k: [c for c in v if c in record(f, k, t)["columns"]]
                                          for k, v in LONG.items()})
        part.insert(1, "frequency", f)
        history.append(part)
history = pd.concat(history, ignore_index=True)
history["start"] = [Period.parse(t).start for t in history["period"]]
history = (history.sort_values(["chip_id", "frequency", "start"]).drop(columns="start")
           .set_index(["chip_id", "frequency", "period"]))
with pd.option_context("display.max_rows", 200, "display.max_columns", None, "display.width", 250):
    display(history.style.format(precision=3, na_rep="missing"))

**What each variable means.** Written from the code that makes it (`plan.py`, `detect.py`,
`assemble_period.py`); the full text is in `panel_pipeline/lib/variables.py`. "Missing when"
matters for the model: a missing value is never a zero.

In [ ]:
present = set(PLAN_COLUMNS) | {"period", "lon", "lat", "amw_key_x", "amw_key_y"}
for layer in ("scores", "postprocess"):
    present |= set(record("quarterly", layer, q)["columns"])
variables = dictionary()
variables = variables[[any(n in present for n in expand(v)) for v in variables["variable"]]]
display(variables.style.hide(axis="index")
        .set_properties(**{"text-align": "left", "white-space": "pre-wrap", "vertical-align": "top"})
        .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))

**How each variable is distributed**, over every chip of `MAP_PERIOD` and of its first month.
For a true / false variable, "mean" is the share that is true. "missing" counts chips without
the value (see "missing when" above). One column is read at a time, so this is light on memory
(about a minute for a period of 46 million chips).

In [ ]:
def describe(frequency: str, tag: str) -> pd.DataFrame:
    out = {}
    for layer in ("scores", "postprocess"):
        path = layer_path(frequency, layer, tag)
        for name in pq.read_schema(path).names:
            if name == "chip_id":
                continue
            a = pq.read_table(path, columns=[name]).column(name)
            n = len(a)
            if pa.types.is_boolean(a.type):
                missing = a.null_count
                true = pc.sum(a).as_py() or 0
                out[name] = {"missing": missing / n, "mean": true / (n - missing) if n > missing else np.nan}
            else:
                v = np.asarray(a.to_numpy(zero_copy_only=False), dtype=np.float64)
                ok = v[np.isfinite(v)]
                out[name] = {"missing": 1 - len(ok) / n, "mean": ok.mean() if len(ok) else np.nan,
                             "sd": ok.std() if len(ok) else np.nan,
                             "min": ok.min() if len(ok) else np.nan,
                             "median": np.median(ok) if len(ok) else np.nan,
                             "max": ok.max() if len(ok) else np.nan}
    return pd.DataFrame(out).T[["missing", "mean", "sd", "min", "median", "max"]]


first_month = Period.parse(q).months()[0].tag
shown = [("quarterly", q)] + ([("monthly", first_month)] if first_month in periods["monthly"] else [])
stats = pd.concat({t: describe(f, t) for f, t in shown}, axis=1)
stats.index.name = "variable"
display(stats.style.format("{:.3f}", na_rep="").format("{:.1%}", subset=[(t, "missing") for _, t in shown]))

In [ ]:
if EXPORT_DIR:
    out = (REPO / EXPORT_DIR).resolve()
    out.mkdir(parents=True, exist_ok=True)
    variables.to_csv(out / "variables.csv", index=False)
    example.to_csv(out / f"example_rows_{q}.csv", index=False)
    history.to_csv(out / "chip_histories.csv")
    stats.to_csv(out / f"distributions_{'_'.join(t for _, t in shown)}.csv")
    print(f"saved to {out}:", ", ".join(sorted(p.name for p in out.glob("*.csv"))))

## 3. Checks

`checks/check_panel.py` reads every period and tests what must always hold:

| check | what must hold |
|---|---|
| rows | every planned chip once, sorted; both layers have the same rows |
| scored | every chip has a score, or its tile is listed as failed |
| votes | `confidence` is the mean of the six votes |
| ranges | scores and shares between 0 and 1, `n_clear` ≤ `n_obs`, nothing infinite |
| flags | `det_raw` is `confidence > 0.4`; stringent ⊂ loose ⊂ raw; flags empty exactly where there is no score |
| distances | a distance exactly for the chips in each catalogue; the geodesic one within 10% of AMW's |
| border | the two scores of each border chip were identical |
| months | a quarter's `n_obs` and `n_clear` are the sums of its three months' |

In [ ]:
if RUN_CHECKS:
    report = check_panel(run, SUBREGIONS)
    summary = report.groupby(["check"], sort=False).agg(periods=("ok", "size"),
                                                        passed=("ok", "sum"),
                                                        example=("detail", "first"))
    print("all checks passed" if report["ok"].all() else "SOME CHECKS FAILED: see below")
    display(summary)
    display(report[~report["ok"]])

## 4. How many chips AMW's rules detect

AMW's three rules, applied to every chip of every period:

* **raw**: `confidence > 0.4`, the model's mean vote.
* **loose** and **stringent**: the *dual threshold*. A chip needs `confidence ≥ 0.43`
  (stringent: 0.55); if it is *isolated*, meaning fewer than 5 other such chips within 3 km,
  it needs `≥ 0.75` (stringent: 0.8). Real mines come in clusters; a lone high score is more
  often a cloud edge or a river bank.

A quarter's image is the median of three months of pictures, so it is usually clearer than
any of its months, and a chip can be detected in the quarter without being detected in a month.
Compare the months with section 5: a month with few clear pictures should show fewer detections.

In [ ]:
rows = []
for f, ts in periods.items():
    for t in ts:
        c = record(f, "postprocess", t)["counts"]
        rows.append({"frequency": f, "period": t, "start": pd.Timestamp(Period.parse(t).start),
                     "raw": c["raw"], "loose": c["loose"]["detected"],
                     "stringent": c["stringent"]["detected"], "Andes": c["andes"],
                     "isolated (loose)": c["loose"]["isolated"]})
counts = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(8, 3.6))
m = counts[counts["frequency"] == "monthly"]
ax.plot(m["start"] + pd.Timedelta(days=14), m["stringent"], color=MONTHLY, lw=2, marker="o",
        ms=6, label="monthly")
quarters = counts[counts["frequency"] == "quarterly"]
for i, (start, n) in enumerate(zip(quarters["start"], quarters["stringent"])):
    ax.plot([start, start + pd.DateOffset(months=3)], [n, n], color=QUARTERLY, lw=2.5,
            label="quarterly (one line across its three months)" if i == 0 else None)
ax.set(title="Chips detected by AMW's stringent rule", ylabel="chips")
ax.set_ylim(bottom=0)
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
plt.show()
counts.drop(columns="start")

## 5. How well each period was seen

A chip's image is the median of the period's **clear** pictures (Cloud Score+ ≥ 0.6, AMW's
cloud test). Where a pixel has no clear picture at all, the image is black there and the
model sees nothing. `n_clear` is the average number of clear pictures per pixel of the chip.

* **no clear picture**: `n_clear = 0`, the whole chip was black to the model;
* **under 1**: on average less than one clear picture per pixel;
* **quality lost**: in the periods scored before 1 November, a chip with *some* black pixels
  lost its `empty_share`, `ndvi_mean`, `bsi_mean`, `cirrus_mean` and `cs_clear_mean` (a bug in
  `detect.py`, fixed since). Its scores are not affected. These values are empty in the panel,
  never wrong.

In [ ]:
rows = []
for f, ts in periods.items():
    for t in ts:
        n_clear = column(f, "scores", t, "n_clear")
        lost = record(f, "scores", t)["masked_pixels"]
        rows.append({"frequency": f, "period": t,
                     "median n_clear": float(np.nanmedian(n_clear)),
                     "no clear picture": float(np.mean(n_clear == 0)),
                     "under 1": float(np.mean(n_clear < 1)),
                     "quality lost": lost["chips_with_lost_quality_values"] / len(n_clear)})
seen = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(8, 3.6))
colours = [MONTHLY if f == "monthly" else QUARTERLY for f in seen["frequency"]]
ax.bar(seen["period"], 100 * seen["under 1"], color=colours, width=0.7)
ax.set(title="Chips with less than one clear picture per pixel", ylabel="% of chips")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=MONTHLY), plt.Rectangle((0, 0), 1, 1, color=QUARTERLY)],
          labels=["monthly", "quarterly"], frameon=False)
plt.show()
seen.style.format({"median n_clear": "{:.1f}", "no clear picture": "{:.1%}",
                   "under 1": "{:.1%}", "quality lost": "{:.1%}"})

## 6. Do detections depend on how well a chip was seen?

If the detector worked equally well under any sky, the share of chips detected would not
change with `n_clear`. A chip with no clear picture at all is a black square and should
never be detected. Pooled over all periods of each frequency:

In [ ]:
edges = [0, 1e-9, 1, 3, 10, np.inf]
labels = ["none (0)", "under 1", "1 to 3", "3 to 10", "10 or more"]
rows = []
for f, ts in periods.items():
    total, hits, top_black = np.zeros(5), np.zeros(5), 0.0
    for t in ts:
        n_clear = column(f, "scores", t, "n_clear")
        conf = column(f, "scores", t, "confidence")
        raw = column(f, "postprocess", t, "det_raw")
        k = np.digitize(np.nan_to_num(n_clear, nan=-1), edges) - 1    # -1: no score
        ok = k >= 0
        total += np.bincount(k[ok], minlength=5)[:5]
        hits += np.bincount(k[ok], weights=raw[ok], minlength=5)[:5]
        top_black = max(top_black, float(np.nanmax(conf[n_clear == 0], initial=0)))
    rows += [{"frequency": f, "n_clear": l, "chips": int(n), "detected (raw)": int(h),
              "share detected": h / n if n else np.nan} for l, n, h in zip(labels, total, hits)]
    print(f"{f}: the highest score of a chip with no clear picture is {top_black:.3f}")
by_seen = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(8, 3.6))
x = np.arange(len(labels))
for i, (f, colour) in enumerate([("monthly", MONTHLY), ("quarterly", QUARTERLY)]):
    d = by_seen[by_seen["frequency"] == f]
    if len(d):
        ax.bar(x + (i - 0.5) * 0.38, 1000 * d["share detected"], width=0.36, color=colour, label=f)
ax.set_xticks(x, labels)
ax.set(title="Raw detections per 1,000 chips, by clear pictures per pixel",
       xlabel="n_clear", ylabel="per 1,000 chips")
ax.legend(frameon=False)
plt.show()
by_seen.style.format({"share detected": "{:.3%}", "chips": "{:,}", "detected (raw)": "{:,}"})

**Reading it.** If the share detected rises with `n_clear`, the detector finds fewer mines
where it sees less, so a monthly count mixes mining with the weather. That matters for any
rule built on months (isolation, open decision D1; persistence, D8), and `n_clear` is
in the panel to control for it.

## 7. Do the months agree with their quarter?

For each quarter whose three months are assembled: of the chips detected in the quarter, how
many were also detected in at least one of its months, and in all three; and the other way
round. The rows line up in every period, so this is a comparison of arrays, row by row.
A mine that is detected in its quarter but not in every month was often just not visible
that month; section 10 shows examples.

In [ ]:
rows = []
for q in periods["quarterly"]:
    months = [m.tag for m in Period.parse(q).months()]
    if not all(m in periods["monthly"] for m in months):
        continue
    for rule in ("det_raw", "det_stringent"):
        in_q = column("quarterly", "postprocess", q, rule)
        in_m = np.stack([column("monthly", "postprocess", m, rule) for m in months])
        any_m, all_m = in_m.any(axis=0), in_m.all(axis=0)
        rows.append({"quarter": q, "rule": rule.removeprefix("det_"),
                     "in quarter": int(in_q.sum()),
                     "...also in a month": (in_q & any_m).sum() / max(in_q.sum(), 1),
                     "...in all 3 months": (in_q & all_m).sum() / max(in_q.sum(), 1),
                     "in a month": int(any_m.sum()),
                     "...also in quarter": (any_m & in_q).sum() / max(any_m.sum(), 1)})
agree = pd.DataFrame(rows)
agree.style.format({c: "{:.1%}" for c in agree.columns if c.startswith("...")})

## 8. Where the detections are

Chips detected by the stringent rule in one quarter, over the subregion outlines.

In [ ]:
ids = pq.read_table(panel / "quarterly" / "postprocess" / f"period={MAP_PERIOD}" / "data.parquet",
                    columns=["chip_id", "det_stringent"])
ids = ids.filter(ids["det_stringent"].fill_null(False))["chip_id"]
squares = ds.dataset(run / "plan" / "chips.parquet").to_table(
    columns=["chip_id", "minx", "miny", "maxx", "maxy"],
    filter=pc.field("chip_id").isin(ids)).to_pandas()
lon, lat = (squares["minx"] + squares["maxx"]) / 2, (squares["miny"] + squares["maxy"]) / 2

fig, ax = plt.subplots(figsize=(8, 6))
for n in sorted(record("quarterly", "scores", MAP_PERIOD)["subregions"]):
    shape = gpd.read_file(REPO / "data" / "boundaries" / "Amazon_ACA" / f"Amazon_ACA_{n}.geojson")
    shape.boundary.plot(ax=ax, color="#8a8984", lw=0.8)
    c = shape.union_all().representative_point()
    ax.annotate(f"subregion {n}", (c.x, c.y), color="#52514e", ha="center")
ax.scatter(lon, lat, s=3, color=QUARTERLY, lw=0)
ax.set(title=f"{len(squares):,} chips detected (stringent), {MAP_PERIOD}",
       xlabel="longitude", ylabel="latitude", aspect="equal")
ax.grid(False)
plt.show()

## 9. AMW's distances and distances on the Earth (open decision D3)

To decide whether a chip is isolated, AMW projects all chips of the period onto **one** flat
map (the UTM zone in the middle of them) and measures there. Far from that zone the map
stretches distances. The panel stores AMW's distance (`kth_km_*`, used by the flags) and the
distance on the Earth's real shape (`kth_km_geo_*`), so the rule can be revisited. The chart
shows the ratio for the loose catalogue of one quarter, by longitude.

In [ ]:
pp = pq.read_table(panel / "quarterly" / "postprocess" / f"period={MAP_PERIOD}" / "data.parquet",
                   columns=["chip_id", "kth_km_loose", "kth_km_geo_loose", "det_loose"]).to_pandas()
cat = pp[pp["kth_km_loose"].notna()]
where = ds.dataset(run / "plan" / "chips.parquet").to_table(
    columns=["chip_id", "minx", "maxx"], filter=pc.field("chip_id").isin(cat["chip_id"])).to_pandas()
cat = cat.merge(where, on="chip_id")
cat["lon"] = (cat["minx"] + cat["maxx"]) / 2
cat["ratio"] = cat["kth_km_loose"] / cat["kth_km_geo_loose"]

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.scatter(cat["lon"], cat["ratio"], s=8, color=MONTHLY, lw=0, alpha=0.5)
ax.axhline(1, color="#8a8984", lw=1)
ax.set(title=f"AMW's distance / distance on the Earth, loose catalogue, {MAP_PERIOD}",
       xlabel="longitude", ylabel="ratio")
plt.show()
iso_amw = cat["kth_km_loose"] > 3.0
iso_geo = cat["kth_km_geo_loose"] > 3.0
print(f"{len(cat):,} chips in the catalogue; ratio {cat['ratio'].min():.3f} to {cat['ratio'].max():.3f}; "
      f"{int((iso_amw != iso_geo).sum())} would change between isolated and not with the geodesic distance")

## 10. One chip through time

Because every period holds the same chips in the same order, row *i* is the same chip in every
file of the panel: following a chip through time needs no join. The five chips with the highest
quarterly score among the stringent detections of the map's quarter, in every period:

In [ ]:
q_ids = column("quarterly", "scores", MAP_PERIOD, "chip_id")
q_conf = column("quarterly", "scores", MAP_PERIOD, "confidence")
q_det = column("quarterly", "postprocess", MAP_PERIOD, "det_stringent")
rows_ = np.argsort(np.where(q_det, q_conf, -1))[::-1][:5]       # positions, same in every file
table = {}
for f, ts in periods.items():
    for t in ts:
        assert np.array_equal(column(f, "scores", t, "chip_id")[rows_], q_ids[rows_])
        conf, n_clear = (column(f, "scores", t, c)[rows_] for c in ("confidence", "n_clear"))
        table[t] = [f"{a:.2f}  ({b:.1f})" for a, b in zip(conf, n_clear)]
print("each cell: confidence  (n_clear, clear pictures per pixel)")
pd.DataFrame(table, index=pd.Index(q_ids[rows_], name="chip_id")).T